# Vinamilk 2026 Charter Scan OCR Diagnostic — Page 8

Compare Tesseract, PaddleOCR, EasyOCR, or Surya against one full-page English physical scan from Vinamilk’s issuer-hosted 2026 Company Charter. Use the private dataset `ltrngnhn/vinamilk-charter-scan-page8-assets`; run one engine per kernel version. The label is a single-annotator visual transcription, so scores are diagnostic only and are not eligible for model selection. Accuracy is comparable only when source/render hashes match; throughput across CPU/GPU kernels is not comparable.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

input_root = Path("/kaggle/input")
for bundle in [*input_root.glob("*/benchmark_project.zip"), *input_root.glob("datasets/*/*/benchmark_project.zip")]:
    extracted = Path("/kaggle/working/benchmark-assets/project")
    shutil.rmtree(extracted.parent, ignore_errors=True)
    shutil.unpack_archive(str(bundle), str(extracted.parent))
project_candidates = {
    *input_root.glob("*/project"),
    *input_root.glob("datasets/*/*/project"),
    *Path("/kaggle/working/benchmark-assets").glob("project"),
    *(entry.parent.parent for entry in input_root.rglob("scripts.processing.evaluate_ocr.py") if entry.parent.name == "scripts"),
}
assets_candidates = sorted(
    project for project in project_candidates
    if (project / "scripts" / "scripts.processing.evaluate_ocr.py").is_file()
)
if len(assets_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one attached OCR benchmark assets/project; "
        f"found: {assets_candidates}"
    )
assets = assets_candidates[0]
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
if repo.exists():
    shutil.rmtree(repo)
shutil.copytree(assets, repo)
%cd /kaggle/working/enterprise-knowledge-ai-agent


In [ ]:
ENGINE = "surya"  # Run one engine per kernel: tesseract, paddleocr, easyocr, or surya.
REQUIRE_GPU = True  # Set True and enable Kaggle GPU for EasyOCR or Surya.
DPI = 180
TESSERACT_PSM = 3
DATASET_PATH = "data/evaluation/vinamilk_ocr_charter_scan_en_v1.json"
DATASET_TAG = "charter-scan-page8-v1"


In [ ]:
import subprocess, sys

# Install one engine only, avoiding incompatible OCR framework combinations.
subprocess.run([sys.executable, "-m", "pip", "install", "PyMuPDF==1.27.2.3", "Pillow==12.1.0"], check=True)
if ENGINE == "paddleocr":
    subprocess.run([sys.executable, "-m", "pip", "--default-timeout=300", "--retries=5", "install", "paddlepaddle==3.3.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cpu/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "paddleocr==3.7.0"], check=True)
elif ENGINE == "easyocr":
    subprocess.run([sys.executable, "-m", "pip", "install", "easyocr==1.7.2"], check=True)
elif ENGINE == "surya":
    subprocess.run([sys.executable, "-m", "pip", "install", "surya-ocr==0.17.1", "transformers==4.57.3"], check=True)
elif ENGINE == "tesseract":
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie"], check=True)
else:
    raise ValueError(f"Unknown ENGINE: {ENGINE}")


In [ ]:
import os

os.environ["HF_HOME"] = "/kaggle/working/model-cache/huggingface"
os.environ["TORCH_HOME"] = "/kaggle/working/model-cache/torch"
if ENGINE == "surya":
    # Fit Surya's detection/recognition batches within Kaggle's 15 GB Tesla T4.
    os.environ["DETECTOR_BATCH_SIZE"] = "16"
    os.environ["RECOGNITION_BATCH_SIZE"] = "128"
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
if ENGINE in {"easyocr", "surya"}:
    import torch
    cuda_available = torch.cuda.is_available()
    print("torch", torch.__version__, "CUDA", cuda_available)
    if REQUIRE_GPU and not cuda_available:
        raise RuntimeError("GPU requested for this OCR engine but CUDA is unavailable; restart with a Kaggle GPU accelerator.")
elif ENGINE == "paddleocr":
    import paddle
    print("paddle", paddle.__version__, "CUDA build", paddle.is_compiled_with_cuda(), "device", paddle.device.get_device())


In [ ]:
import json
dataset_path = Path(DATASET_PATH)
if not dataset_path.is_file():
    raise FileNotFoundError(f"Missing OCR labels: {dataset_path}")
dataset = json.loads(dataset_path.read_text(encoding="utf-8"))
for source in sorted({page["file"] for page in dataset["pages"]}):
    if not Path(source).is_file():
        raise FileNotFoundError(f"Missing benchmark source PDF: {source}")
output = Path("/kaggle/working") / f"vinamilk-ocr-{DATASET_TAG}-{ENGINE}.json"
command = [sys.executable, "scripts/processing/evaluate_ocr.py", "--dataset", DATASET_PATH, "--engines", ENGINE, "--dpi", str(DPI), "--output", str(output)]
if ENGINE == "tesseract":
    command.extend(["--tesseract-psm", str(TESSERACT_PSM)])
subprocess.run(command, check=True)
print(f"Download the result from Kaggle Outputs: {output}")


## Limitations

This is one English legal-document scan page, from one source PDF, with a single-annotator visual reference that has not been independently reviewed. It does not expand Vietnamese scan coverage or support an OCR winner claim. The page hash and engine/runtime configuration in each report must match before comparing outputs. PaddleOCR runs on CPU in this notebook; EasyOCR and Surya use a Kaggle GPU when enabled. Do not compare their throughput across unlike hardware.
